In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dt = pd.read_csv('../data/processed/train_data.csv')

In [2]:
object_columns = dt.select_dtypes(include=['object']).columns

for col in object_columns:
    dt[col] = dt[col].str.lower().str.strip()

print("Log: Text standardization (lowercase & strip) completed for all object columns.")

Log: Text standardization (lowercase & strip) completed for all object columns.


In [3]:
hidden_missing_values = ['unknown', 'none', 'not defined', 'null', 'nan', '']
dt[object_columns] = dt[object_columns].replace(hidden_missing_values, np.nan)

print(f"Log: Hidden missing values {hidden_missing_values} converted to actual np.nan.")

Log: Hidden missing values ['unknown', 'none', 'not defined', 'null', 'nan', ''] converted to actual np.nan.


In [4]:
dt.describe(include='object').T

,count,unique,top,freq
ProductCD,590540,5,w,439670
card4,588963,4,visa,384767
card6,588969,4,debit,439938
P_emaildomain,496084,59,gmail.com,228355
R_emaildomain,137291,60,gmail.com,57147
M4,309096,3,m0,196405
id_23,5169,3,ip_proxy:transparent,3489
id_30,77565,75,windows 10,21155
id_31,140282,130,chrome 63.0,22000
id_33,73289,260,1920x1080,16874


In [5]:
#Identify numeric columns
numeric_cols = dt.select_dtypes(include=['int64', 'float64']).columns

#Find potential categorical columns based on unique value count
potential_categorical = []
threshold = 10

for col in numeric_cols:
    unique_count = dt[col].nunique()
    
    #Exclude the target variable from this list to prevent data leakage
    if unique_count < threshold and col != 'isFraud':
        potential_categorical.append(col)

print(f"Found {len(potential_categorical)} disguised categorical columns.")

#Convert them to 'category' data type
dt[potential_categorical] = dt[potential_categorical].astype('category')

print("Type casting completed. Memory usage optimized.")

Found 105 disguised categorical columns.
Type casting completed. Memory usage optimized.


In [6]:
dt['TransactionAmt_Log'] = np.log1p(dt['TransactionAmt'])
dt.drop('TransactionAmt', axis=1, inplace=True)

In [6]:
def process_time_features(df):

    print("Extracting time features...")
    df['Transaction_Hour'] = np.floor((df['TransactionDT'] % 86400) / 3600).astype(int)
    
    # Calculate the day of the week (0 to 6)
    df['Transaction_DayOfWeek'] = np.floor((df['TransactionDT'] / 86400) % 7).astype(int)
    
    # Drop the original TransactionDT column 
    # Prevents models from inferring false linear trends from a continuously increasing counter
    df = df.drop('TransactionDT', axis=1)
    
    print("Process completed! 'Transaction_Hour' and 'Transaction_DayOfWeek' added.")
    print("Original 'TransactionDT' column dropped.")
    
    return df


dt = process_time_features(dt)
print(dt[['Transaction_Hour', 'Transaction_DayOfWeek']].head())

Extracting time features...
Process completed! 'Transaction_Hour' and 'Transaction_DayOfWeek' added.
Original 'TransactionDT' column dropped.
   Transaction_Hour  Transaction_DayOfWeek
0                 0                      1
1                 0                      1
2                 0                      1
3                 0                      1
4                 0                      1


In [7]:
# Define exception columns to keep even if they exceed 70% missing ratio
d_cols_to_keep = ['D7', 'D12', 'D13', 'D14', 'D6', 'D8', 'D9']
id_nums = ['04', '03', '10', '09', '13', '16', '06', '05', '20', '19', 
           '17', '31', '15', '28', '29', '35', '37', '36', '38', '11', '02', '01', '12']
id_cols_to_keep = [f"id_{num}" for num in id_nums]

# Add M6 to the explicit exception list
m_cols_to_keep = ['M6']

# Merged exception set
exception_list = set(d_cols_to_keep + id_cols_to_keep + m_cols_to_keep)

print(f"Exception list prepared with {len(exception_list)} columns.")

Exception list prepared with 31 columns.


In [8]:
def get_fraud_diff(df, col_name, target='isFraud'):
    """
    Calculates the absolute difference in fraud rates between 
    missing and non-missing values for a given column.
    """
    missing_mask = df[col_name].isna()
    
    if missing_mask.sum() == 0 or (~missing_mask).sum() == 0:
        return 0
        
    fraud_missing = df.loc[missing_mask, target].mean() * 100
    fraud_filled  = df.loc[~missing_mask, target].mean() * 100
    
    return abs(fraud_missing - fraud_filled)

def get_smart_impute_value(df, col_name):
    """
    Automatically decides the best imputation value for a numeric column
    based on its distribution (unique values and skewness).
    """
    series = df[col_name]
    
    # If not numeric, fallback to mode
    if not pd.api.types.is_numeric_dtype(series):
        return series.mode()[0] if not series.mode().empty else "Unknown"
    
    n_unique = series.nunique()
    skewness = series.skew()
    
    # Logic 1: If very few unique values (e.g., < 20), treat as categorical/flag -> Use Mode
    if n_unique < 20:
        return series.mode()[0]
        
    # Logic 2: If highly skewed (lots of 0.0s or extreme outliers like 13422) -> Use Median
    # Skewness > 1 or < -1 indicates a highly skewed distribution
    elif abs(skewness) > 1.0:
        return series.median()
        
    # Logic 3: If somewhat normally distributed -> Use Mean (or Median as a safe default)
    else:
        return series.mean()

In [9]:
# Constants
MISSING_THRESHOLD = 70.0
FRAUD_DIFF_THRESHOLD = 5.0
TARGET = 'isFraud'

# Calculate missing ratios
missing_ratios = (dt.isna().sum() / len(dt)) * 100
high_missing_cols = missing_ratios[missing_ratios > MISSING_THRESHOLD].index

cols_to_drop = ['TransactionID']
cols_to_flag = []

for col in high_missing_cols:
    if col == TARGET:
        continue
        
    # Condition 1: Check exceptions (D columns, IDs, and M6)
    if col in exception_list:
        cols_to_flag.append(col)
        
    # Condition 2: Check V columns for fraud difference
    elif str(col).startswith('V'):
        diff = get_fraud_diff(dt, col, target=TARGET)
        if diff >= FRAUD_DIFF_THRESHOLD:
            cols_to_flag.append(col)
        else:
            cols_to_drop.append(col)
            
    # Condition 3: Others above threshold are dropped
    else:
        cols_to_drop.append(col)

# Execute dropping
dt.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f"Dropped {len(cols_to_drop)} columns.")
print(f"Kept {len(cols_to_flag)} high-missing columns as exceptions (to be flagged).")

Dropped 67 columns.
Kept 142 high-missing columns as exceptions (to be flagged).


In [ ]:
# SAFE FILLNA FUNCTION
def safe_fillna(df, col, fill_val):
   # If the column is of categorical type, add the new value to the categories first
    if isinstance(df[col].dtype, pd.CategoricalDtype):
        if fill_val not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories([fill_val])
            
    df[col] = df[col].fillna(fill_val)


# 1. ADDING FLAGS TO RECOVERED COLUMNS
for col in cols_to_flag:
    dt[f"{col}_is_missing"] = dt[col].isna().astype(int)

# 2. ANOMALY IMPUTATION: Filling high-null exception columns with -999 / 'Missing_Special'
print("Filling high-null exception columns with -999 / 'Missing_Special'...")
for col in cols_to_flag:
    if pd.api.types.is_numeric_dtype(dt[col]) or (isinstance(dt[col].dtype, pd.CategoricalDtype) and pd.api.types.is_numeric_dtype(dt[col].cat.categories)):
        safe_fillna(dt, col, -999)
    else:
        safe_fillna(dt, col, "Missing_Special")

# 3. NORMAL IMPUTATION: Remaining columns (with < 70% nulls)
print("Rule-based imputation starting for remaining columns...")

for col in dt.columns:
    if col == TARGET or str(col).endswith('_is_missing') or col in cols_to_flag:
        continue
        
    if dt[col].isna().sum() > 0:
        
        # Rule A: D columns -> Median
        if str(col).startswith('D'):
            if pd.api.types.is_numeric_dtype(dt[col]):
                fill_val = dt[col].median()
            else:
                fill_val = dt[col].mode()[0]
                
        # Rule B: M columns -> Mode
        elif str(col).startswith('M'):
            if not dt[col].mode().empty:
                fill_val = dt[col].mode()[0]
            else:
                fill_val = "Unknown"
                
        # Rule C: ID columns -> Smart Imputation
        elif str(col).startswith('id_'):
            fill_val = get_smart_impute_value(dt, col)
            
        # Rule D: Everything else (V columns etc.)
        else:
            if pd.api.types.is_numeric_dtype(dt[col]):
                fill_val = dt[col].median()
            else:
                fill_val = dt[col].mode()[0] if not dt[col].mode().empty else "Unknown"
        
        safe_fillna(dt, col, fill_val)

print("All missing values (including categorical types) have been successfully filled.")

In [11]:
dt.head(5)

,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,id_17_is_missing,id_19_is_missing,id_20_is_missing,id_28_is_missing,id_29_is_missing,id_31_is_missing,id_35_is_missing,id_36_is_missing,id_37_is_missing,id_38_is_missing
0,0,68.5,w,13926,361.0,150.0,discover,142.0,credit,315.0,...,1,1,1,1,1,1,1,1,1,1
1,0,29.0,w,2755,404.0,150.0,mastercard,102.0,credit,325.0,...,1,1,1,1,1,1,1,1,1,1
2,0,59.0,w,4663,490.0,150.0,visa,166.0,debit,330.0,...,1,1,1,1,1,1,1,1,1,1
3,0,50.0,w,18132,567.0,150.0,mastercard,117.0,debit,476.0,...,1,1,1,1,1,1,1,1,1,1
4,0,50.0,h,4497,514.0,150.0,mastercard,102.0,credit,420.0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# --- MISSING VALUE ANALYSIS ---
print("--- MISSING VALUE ANALYSIS ---")

# Calculate missing value percentages for all columns
missing_values = dt.isnull().sum()
missing_percentages = (missing_values / len(dt)) * 100

# Create a DataFrame for better viewing
missing_df = pd.DataFrame({'Missing_Count': missing_values, 'Missing_Percentage': missing_percentages})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Percentage', ascending=False)

print(f"Total columns with missing values: {len(missing_df)} out of {dt.shape[1]}")

--- MISSING VALUE ANALYSIS ---
Total columns with missing values: 0 out of 510


In [13]:
def apply_one_hot_encoding(df):
    """
    Applies One-Hot Encoding to low-cardinality categorical features BEFORE the train/test split.
    Does not cause data leakage as it does not rely on the target variable.
    """
    print("Log: Starting One-Hot Encoding phase...")
    encoded_df = df.copy()
    
    ohe_cols = [
        'ProductCD', 'card4', 'card6', 'V10', 'V12', 'V27', 'V28', 
        'V35', 'V75', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122'
    ]
    
    valid_ohe_cols = [col for col in ohe_cols if col in encoded_df.columns]
    print(f"Log: Applying OHE to {len(valid_ohe_cols)} columns...")
    
    # drop_first=True prevents the dummy variable trap (multicollinearity)
    encoded_df = pd.get_dummies(encoded_df, columns=valid_ohe_cols, drop_first=True)
    
    print("Log: One-Hot Encoding completed.")
    return encoded_df

df = apply_one_hot_encoding(dt)

Log: Starting One-Hot Encoding phase...
Log: Applying OHE to 15 columns...
Log: One-Hot Encoding completed.


In [14]:
df.head(5)

,isFraud,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,P_emaildomain,...,V119_3.0,V120_1.0,V120_2.0,V120_3.0,V121_1.0,V121_2.0,V121_3.0,V122_1.0,V122_2.0,V122_3.0
0,0,68.5,13926,361.0,150.0,142.0,315.0,87.0,19.0,gmail.com,...,False,True,False,False,True,False,False,True,False,False
1,0,29.0,2755,404.0,150.0,102.0,325.0,87.0,8.0,gmail.com,...,False,True,False,False,True,False,False,True,False,False
2,0,59.0,4663,490.0,150.0,166.0,330.0,87.0,287.0,outlook.com,...,False,True,False,False,True,False,False,True,False,False
3,0,50.0,18132,567.0,150.0,117.0,476.0,87.0,8.0,yahoo.com,...,False,True,False,False,True,False,False,True,False,False
4,0,50.0,4497,514.0,150.0,102.0,420.0,87.0,8.0,gmail.com,...,False,True,False,False,True,False,False,True,False,False


In [15]:
import pandas as pd

def clean_and_encode_group2_3(df):
    """
    Group 2: Extracts email domains (e.g., .com) and applies One-Hot Encoding.
    Group 3: Cleans IT features (removes versions) to prepare for Target Encoding.
    """
    print("Log: Starting Group 2 (Email) and Group 3 (IT) Preprocessing...")
    df_clean = df.copy()

    email_cols = ['P_emaildomain', 'R_emaildomain']
    ext_cols_created = []
    
    for col in email_cols:
        if col in df_clean.columns:
            # Extract the part after the last dot (e.g., 'gmail.com' -> 'com')
            # If there is no dot (like 'Unknown'), it will just keep the word
            df_clean[f'{col}_ext'] = df_clean[col].astype(str).str.split('.').str[-1]
            ext_cols_created.append(f'{col}_ext')
            
            df_clean = df_clean.drop(columns=[col])

    # Apply One-Hot Encoding to the newly created extension columns
    if ext_cols_created:
        print(f"Log: Applying OHE to email extensions: {ext_cols_created}")
        df_clean = pd.get_dummies(df_clean, columns=ext_cols_created, drop_first=True)
    
    # id_30 (OS): Extract the base OS (e.g., 'windows 10' -> 'windows')
    if 'id_30' in df_clean.columns:
        print("Log: Cleaning 'id_30' (OS)...")
        df_clean['id_30_base'] = df_clean['id_30'].astype(str).str.split(' ').str[0]
        df_clean = df_clean.drop(columns=['id_30'])

    # id_31 (Browser): Keep only letters, remove numbers (e.g., 'chrome 63.0' -> 'chrome')
    if 'id_31' in df_clean.columns:
        print("Log: Cleaning 'id_31' (Browser)...")
        df_clean['id_31_base'] = df_clean['id_31'].astype(str).str.replace(r'[^a-zA-Z]', '', regex=True)
        df_clean = df_clean.drop(columns=['id_31'])

    # DeviceInfo: Extract brand name (e.g., 'samsung sm-g892a' -> 'samsung')
    if 'DeviceInfo' in df_clean.columns:
        print("Log: Cleaning 'DeviceInfo' (Brands)...")
        df_clean['device_brand'] = df_clean['DeviceInfo'].astype(str).str.split(r'[ -]').str[0]
        df_clean = df_clean.drop(columns=['DeviceInfo'])

    print("Log: Preprocessing for Group 2 & 3 completed.")
    return df_clean


df = clean_and_encode_group2_3(df)

Log: Starting Group 2 (Email) and Group 3 (IT) Preprocessing...
Log: Applying OHE to email extensions: ['P_emaildomain_ext']
Log: Cleaning 'id_31' (Browser)...
Log: Preprocessing for Group 2 & 3 completed.


In [16]:
df.head(5)

,isFraud,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,C1,...,V122_3.0,P_emaildomain_ext_de,P_emaildomain_ext_es,P_emaildomain_ext_fr,P_emaildomain_ext_gmail,P_emaildomain_ext_jp,P_emaildomain_ext_mx,P_emaildomain_ext_net,P_emaildomain_ext_uk,id_31_base
0,0,68.5,13926,361.0,150.0,142.0,315.0,87.0,19.0,1.0,...,False,False,False,False,False,False,False,False,False,MissingSpecial
1,0,29.0,2755,404.0,150.0,102.0,325.0,87.0,8.0,1.0,...,False,False,False,False,False,False,False,False,False,MissingSpecial
2,0,59.0,4663,490.0,150.0,166.0,330.0,87.0,287.0,1.0,...,False,False,False,False,False,False,False,False,False,MissingSpecial
3,0,50.0,18132,567.0,150.0,117.0,476.0,87.0,8.0,2.0,...,False,False,False,False,False,False,False,False,False,MissingSpecial
4,0,50.0,4497,514.0,150.0,102.0,420.0,87.0,8.0,1.0,...,False,False,False,False,False,False,False,False,False,samsungbrowser


In [18]:
df.to_csv('../data/processed/train_cleaned.csv', index=False)